processing results of [farneback.py](../src/farneback.py)

Mac: `femba` Micromamba env

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from typing import Final

import numpy as np
import matplotlib.pyplot as plt

from src.utils_spectral_power import smoothen


data_dir = Path("data/farneback/subj1_cam1")
video_fname = "28-MAY-2025_00h04m33.969s.mp4"
video_path = data_dir / video_fname

times_path = data_dir / f"{video_path.stem}_times.txt"
times_np = np.loadtxt(times_path) # times in seconds, 1D array
time_delta: Final[float] = times_np[1] - times_np[0]

In [ ]:
from src.utils_spectral_power import load_and_merge_intensities
from matplotlib.ticker import FixedLocator, FixedFormatter


path_to_merge = data_dir / "demo_merge"
times_all, intensities_all, timestamp_pairs_all = load_and_merge_intensities(path_to_merge)
# plt.plot(times_all, intensities_all)

n_points_smoothened, intensities_all_smoothed = smoothen(intensities_all, 15, time_delta)
times_smoothed = times_all[n_points_smoothened:]

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(times_smoothed, intensities_all_smoothed)

# Use each video's start as a tick; label with the parsed timestamp string
tick_positions = [sec for _, sec in timestamp_pairs_all]
tick_labels    = [ts  for ts, _ in timestamp_pairs_all]

ax.xaxis.set_major_locator(FixedLocator(tick_positions))
ax.xaxis.set_major_formatter(FixedFormatter(tick_labels))
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import find_peaks

threshold_sec = 84573.739  # 23h29m33.739s = 84573.739
mask = times_smoothed > threshold_sec
times_trimmed = times_smoothed[mask]
intensities_trimmed = np.array(intensities_all_smoothed)[mask]

peak_indices, _ = find_peaks(intensities_trimmed, prominence=0.0)
print(f"Found {len(peak_indices)} peaks")

peak_times = times_trimmed[peak_indices]
peak_values = intensities_trimmed[peak_indices]

def plot_peaks(_peak_values: np.ndarray, title: str = "Distribution of peak intensities") -> None:
    _, ax = plt.subplots(figsize=(3, 4))
    ax.scatter(
        np.zeros_like(_peak_values) + np.random.uniform(-0.1, 0.1, size=_peak_values.size),
        _peak_values,
        s=6,
        alpha=0.6,
        color="C0",
        zorder=3,
    )
    ax.set_xticks([0])
    ax.set_xticklabels(["peak values"])
    ax.set_ylabel("intensity")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_peaks(peak_values)

In [ ]:
intensity_threshold = 0.25
strong = peak_values > intensity_threshold
strong_times  = peak_times[strong]
strong_values = peak_values[strong]

plot_peaks(strong_values, f"strong peaks: > {intensity_threshold}")

In [ ]:
order = np.argsort(peak_values)[::-1]  # descending
top_n = 30
top_indices = order[:top_n]

top_times = peak_times[top_indices]
top_values = peak_values[top_indices]

# Keep them in chronological order for readability
chrono = np.argsort(top_times)
top_times = top_times[chrono]
top_values = top_values[chrono]

print(f"Top {top_n} peaks (time [s], time [h:m:s], value):")
for t, v in zip(top_times, top_values):
    h = int(t // 3600) % 24
    m = int((t % 3600) // 60)
    s = t % 60
    print(f"  {t:10.3f}  {h:02d}h{m:02d}m{s:06.3f}s  {v:.4f}")


In [ ]:
# Cluster by time (gap-based grouping) ---
max_gap_sec = 30.0  # peaks closer than this belong to the same cluster

order = np.argsort(strong_times)
t_sorted = strong_times[order]
v_sorted = strong_values[order]

gaps = np.diff(t_sorted)
labels_sorted = np.concatenate(([0], np.cumsum(gaps > max_gap_sec)))

clusters = []
for lab in np.unique(labels_sorted):
    member = labels_sorted == lab
    t = t_sorted[member]
    v = v_sorted[member]
    clusters.append({
        "label": int(lab),
        "times": t,
        "values": v,
        "median_intensity": float(np.median(v)),
        "max_intensity": float(np.max(v)),
        "t_start": float(t.min()),
        "t_end": float(t.max()),
        "t_peak": float(t[np.argmax(v)]),
        "n_peaks": int(member.sum()),
    })

print(f"{len(clusters)} clusters from {strong_times.size} strong peaks "
      f"(max gap = {max_gap_sec}s)")

In [ ]:
top_n = 15
top_clusters = sorted(clusters, key=lambda c: c["median_intensity"], reverse=True)[:top_n]
top_clusters_chrono = sorted(top_clusters, key=lambda c: c["t_start"])

def fmt_time(t: float) -> str:
    h = int(t // 3600) % 24
    m = int((t % 3600) // 60)
    s = t % 60
    return f"{h:02d}h{m:02d}m{s:06.3f}s"

print(f"\nTop {len(top_clusters)} clusters by median intensity:")
for c in top_clusters_chrono:
    print(
        f"  {c['t_start']:10.3f}  {fmt_time(c['t_start'])} – {fmt_time(c['t_end'])}  "
        f"n={c['n_peaks']:2d}  median={c['median_intensity']:.4f}  "
        f"max={c['max_intensity']:.4f}  peak@{fmt_time(c['t_peak'])}"
    )

In [ ]:
import json

clusters_json = []
for c in top_clusters_chrono:
    clusters_json.append({
        "label": c["label"],
        "t_start": c["t_start"],
        "t_end": c["t_end"],
        "t_peak": c["t_peak"],
        "n_peaks": c["n_peaks"],
        "median_intensity": c["median_intensity"],
        "max_intensity": c["max_intensity"],
        "times": c["times"].tolist(),
        "values": c["values"].tolist(),
    })

out_path = path_to_merge / "top_clusters.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(clusters_json, f, indent=2, ensure_ascii=False)

print(f"Saved {len(clusters_json)} clusters to {out_path}")